# exp_softgate — weighted retrieval-blend, attack the TREC22 oracle headroom (0.627 > SOTA)

Hard per-topic routing (gate v2) couldn't capture the TREC22 oracle (0.6269 > h2oloo 0.6125) — no
label-free signal flags the topics the ensemble buries. This tries the **soft** version: re-inject the
retrieval order as a weighted RRF blend of ensemble-rank + retrieval-rank, `blend = 1/(k+ens_rank) +
w/(k+ret_rank)`. LambdaMART (trained on 2021, where reranking helps) down-weights raw retrieval; a
blend weight `w` tuned on the **develop** set (TREC21+KZ) re-injects it uniformly — recovering buried
eligibles without needing to identify which topics they're in. CPU-only, cached scores.

`w=0` = pure ensemble (current 0.575 on TREC22). Larger `w` -> retrieval dominates. Question: does a
develop-tuned `w>0` push TREC22 above 0.6125? Caveat: TREC21/KZ ensemble is in-sample (§2h) so develop
tuning is biased toward `w=0` — if it STILL prefers `w>0` that's a robust signal; the TREC22-argmax
`w` is also shown as the ceiling.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q datasets pytrec_eval lightgbm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np, pytrec_eval, lightgbm as lgb
from ctmatch.experiments import ExperimentConfig, load_eval
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')
booster = lgb.Booster(model_file=cfg.path('models/ensemble_nqs.txt'))
FEAT = json.load(open(cfg.path('models/ensemble_nqs_features.json')))

# ── load all-source cached features (trec21/kz/trec22) + reconstruct ensemble rankings ──
SRC = ['trec21', 'kz', 'trec22']
sets = load_eval(cfg, SRC)
pool = json.load(open(cfg.path('data/pool_nqs.json')))
rfeat = {}
for l in open(cfg.path('data/retrieval_feats_nqs.jsonl')):
    r = json.loads(l); rfeat[(r['source'], r['topic_id'], r['doc_id'])] = r
def _load(name, key):
    d = {}; p = cfg.feat_file(name)
    if os.path.exists(p):
        for l in open(p):
            r = json.loads(l); d[(r['source'], r['topic_id'], r['doc_id'])] = r[key]
    return d
llm = _load('llm_scores', 'llm_score'); topi = _load('topicality', 'topicality')
cm = {}
cmp = cfg.feat_file('condition_match_exp')
if os.path.exists(cmp):
    for l in open(cmp):
        r = json.loads(l); cm[(r['source'], r['topic_id'], r['doc_id'])] = r['condition_match']
ce = {}
for tag, path in [('clf', cfg.ce_cache_path('clf_R')), ('clft', cfg.ce_cache_path('clf_topic'))]:
    if os.path.exists(path): ce[tag] = np.load(path, allow_pickle=True)['d'].item()
def featvec(s, t, d):
    rf = rfeat.get((s, t, d), {}); cr, cp = ce.get('clf', {}).get((s, t, d), (0., 0.))
    tr, tp = ce.get('clft', {}).get((s, t, d), (0., 0.))
    v = {'bm25': rf.get('bm25', 0.), 'bm25_rank': rf.get('bm25_rank', cfg.cand_k), 'dense': rf.get('dense', 0.),
         'dense_rank': rf.get('dense_rank', cfg.cand_k), 'rrf': rf.get('rrf', 0.), 'clf_rel': cr, 'clf_partial': cp,
         'clf_topic_rel': tr, 'clf_topic_partial': tp, 'llm_yesno': llm.get((s, t, d), cfg.llm_floor),
         'topicality': topi.get((s, t, d), 0.), 'condition_match': cm.get((s, t, d), 0.)}
    return [v[f] for f in FEAT]

ens, ret, rel = {}, {}, {}
for s in SRC:
    for t in pool[s]:
        if t not in sets[s]['rel_dict'] or t not in sets[s]['topic2text']: continue
        docs = [d for d in pool[s][t] if (s, t, d) in rfeat]
        if not docs: continue
        X = np.array([featvec(s, t, d) for d in docs], dtype=np.float32)
        ens[(s, t)] = {d: float(p) for d, p in zip(docs, booster.predict(X))}
        ret[(s, t)] = {d: rfeat[(s, t, d)]['rrf'] for d in docs}
        rel[(s, t)] = {d: int(r) for d, r in sets[s]['rel_dict'][t].items()}

# ── TREC23 (held-out; ensemble from the saved run) ──
topics23 = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
q23 = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); q23.setdefault(t, {})[d] = int(r)
pool23 = json.load(open(f'{T23}/pool_nqs_2023.json'))
rrf23 = {}
for l in open(f'{T23}/retrieval_feats_2023.jsonl'):
    r = json.loads(l); rrf23.setdefault(r['topic_id'], {})[r['doc_id']] = r['rrf']
for l in open(f'{T23}/run_ext2023.txt'):
    t, _, d, rk, sc, _tag = l.split()
    if t in q23:
        ens.setdefault(('trec23', t), {})[d] = float(sc); ret.setdefault(('trec23', t), {}).setdefault(d, rrf23.get(t, {}).get(d, -1e9)); rel[('trec23', t)] = {dd: int(rr) for dd, rr in q23[t].items()}
for t in q23:
    ret[('trec23', t)] = {d: rrf23.get(t, {}).get(d, -1e9) for d in pool23[t]}
print('reconstructed rankings for', len({k[0] for k in ens}), 'datasets')

In [ ]:
# Weighted RRF blend of ensemble-rank + retrieval-rank; NDCG@10 per (source) via pytrec.
def ndcg_blend(w, keys, K=60):
    run, qrels = {}, {}
    for (s, t) in keys:
        docs = list(ens[(s, t)])
        er = {d: i for i, d in enumerate(sorted(docs, key=lambda d: ens[(s, t)][d], reverse=True))}
        rr = {d: i for i, d in enumerate(sorted(docs, key=lambda d: ret[(s, t)].get(d, -1e9), reverse=True))}
        run[f'{s}:{t}'] = {d: 1.0/(K+er[d]+1) + w/(K+rr[d]+1) for d in docs}
        qrels[f'{s}:{t}'] = {d: rel[(s, t)][d] for d in docs}
    ev = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run)
    return float(np.mean([v['ndcg_cut_10'] for v in ev.values()]))

keys = {s: [k for k in ens if k[0] == s] for s in ['trec21', 'kz', 'trec22', 'trec23']}
dev = keys['trec21'] + keys['kz']
WS = [0, 0.1, 0.25, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0, 100.0]
dev_curve = {w: ndcg_blend(w, dev) for w in WS}
wstar = max(WS, key=lambda w: dev_curve[w])
w22 = max(WS, key=lambda w: ndcg_blend(w, keys['trec22']))   # TREC22-argmax = ceiling of this blend
print('develop (TREC21+KZ) NDCG@10 by blend weight w:')
for w in WS: print(f'   w={w:<6} {dev_curve[w]:.4f}{"  <- develop-optimal" if w==wstar else ""}')
print(f'\n{"dataset":9s} {"w=0 (ens)":>10s} {"w*="+str(wstar):>12s} {"w22(ceiling)":>14s} {"w=inf (ret)":>12s}')
for s in ['trec22', 'trec23', 'trec21', 'kz']:
    row = [ndcg_blend(w, keys[s]) for w in [0, wstar, w22, 100.0]]
    print(f'{s:9s} {row[0]:>10.4f} {row[1]:>12.4f} {row[2]:>14.4f} {row[3]:>12.4f}')
print(f'\nh2oloo TREC22 = 0.6125.  develop-tuned w*={wstar} -> TREC22 {ndcg_blend(wstar, keys["trec22"]):.4f}')
print('If develop-w* pushes TREC22 above 0.6125 (a develop-selected, not test-fit, weight) -> clean SOTA-beat lead.')